# Setup

In [ ]:
# base
from pathlib import Path
import os
import sys
import gc
import re
import warnings
import logging
import pickle
from time import ctime, time
from datetime import timedelta
from collections import Counter

# data manipulation
import numpy as np
import pandas as pd
import itables
import seaborn as sns
import matplotlib.pyplot as plt

# single cell
import anndata as ad
import scanpy as sc
import liana as li

# custom
from single_cell.R import *
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from utils import *

itables.init_notebook_mode(connected=True)  # Use connected=False for offline use
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", pd.errors.DtypeWarning)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
mlogger = logging.getLogger("matplotlib")
mlogger.setLevel(logging.WARNING)

CORES = 10
DATADIR = Path("../../../../data")
OUTSDIR = Path("../../../../outs")
REFDIR = Path("../../../../references")
MAIN_DIR = DATADIR / "processed" / "single_cell" / "combined"
SUBSETS_DIR = MAIN_DIR / "subsets"

METADATA = ["Diet", "Age", "Depot", "Sex"]
DOUBLETMETHODS = ["scDblFinder", "DoubletFinder", "doubletdetection", "scrublet"]
INT_KEY = "INT_harmony-Identifier"

%matplotlib inline
# R_preload()
mpl.rcdefaults()
gc.collect()

2769

# Liana Aggegate Data

In [6]:
import omnipath as op
op.interactions.AllInteractions.get().to_csv(REFDIR / "omnipath_all.csv")

In [ ]:
adata = sc.read_h5ad(MAIN_DIR / "All_Data.h5ad")

In [ ]:
# ligand-receptor pairs
ccc_db = pd.read_csv(REFDIR / "ethan_ccc_database.csv")
cell2cell_interactions(adata, "celltype", ccc_db, cores=CORES)

# for cond in adata_clean.obs["CondDietition"].cat.categories:
#     tmp = adata_clean[adata_clean.obs["Condition"] == cond]
#     cell2cell_interactions(tmp, "celltype", ccc_db, cores=CORES)
#     adata.uns["ccc_filtered_{cond}"] = tmp.uns ["ccc_filtered"]

adata

Processing mouseconsensus: 100%|██████████| 18/18 [00:16<00:00,  1.09it/s] 


found lowercase letters in first cell 'Dll1' of 'mouseconsensus', not converting!


Using the `normalized` layer!


Using provided `resource`.


0.06 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 288258 samples and 2012 features
Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


  2%|▏         | 19/1000 [00:00<00:15, 62.91it/s]/mnt/DATA/home/ethung/spatial_seq/.venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/mnt/DATA/home/ethung/spatial_seq/.venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/mnt/DATA/home/ethung/spatial_seq/.venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. Se

Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


AnnData object with n_obs × n_vars = 288258 × 29069
    obs: 'Identifier', 'n_genes', 'Dataset', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'Diet', 'Age', 'Sex', 'Depot', 'celltype_So2025', 'celltypist_So2025', 'celltype_Emont2022', 'celltypist_Emont2022', 'celltype'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'PCA', 'PCA_hvg', 'UMAP_INT_harmony-Dataset', 'UMAP_INT_harmony-Identifier', 'UMAP_INT_harmony_hvg-Dataset', 'UMAP_INT_harmony_hvg-Identifier', 'UMAP_INT_none', 'UMAP_INT_none_hvg', 'UMAP_INT_scanorama-Dataset', 'UMAP_INT_scanorama-Identifier', 'UMAP_INT_scanorama_hvg-Dataset', 'UMAP_INT_scanorama_hvg-Identifier', 'hvg', 'log1p', 'methods', 'neighbors', 'neighbors_INT_harmony-Dataset', 'neighbors_INT_harmony-Iden

# MOFA
see [liana tutorial](https://liana-py.readthedocs.io/en/latest/notebooks/mofacellular.html#fitting-a-mofa-model)